# Train RSSM World Model on MiniGrid

Training a Recurrent State-Space Model (RSSM) from DreamerV2 on MiniGrid-Empty-16x16-v0.

Based on the repository: https://github.com/mihalko711/tbank_intro_problem

## 1. Setup

Install dependencies and clone the repository.

In [ ]:
!pip install gymnasium minigrid matplotlib pyyaml Pillow tqdm -q

import sys
!git clone https://github.com/mihalko711/tbank_intro_problem.git
sys.path.insert(0, 'tbank_intro_problem')
%cd tbank_intro_problem

## 2. Imports, config, initialization

In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output
from tqdm.notebook import tqdm

from src import (
    RSSMWorldModel, collect_episode, evaluate,
    get_env_properties, make_minigrid_env, seed_everything,
)
from src.policy import ScriptedPolicy

with open('configs/minigrid_default.yml') as f:
    config = yaml.safe_load(f)
rssm_cfg = config['rssm']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
seed_everything(config['seed'])

env = make_minigrid_env(config['environment_name'], seed=config['seed'])
env_eval = make_minigrid_env(config['environment_name'], seed=config['seed'] + 999)
obs_shape, action_size = get_env_properties(env)
print(f'Obs shape: {obs_shape}, action size: {action_size}')

rssm = RSSMWorldModel(obs_shape, action_size, rssm_cfg, device)
scripted_policy = ScriptedPolicy(env, action_size, epsilon=0.05, device=device)

## 3. Initial data collection

Collect episodes using a random policy to fill the replay buffer.

In [ ]:
print(f"Collecting {config['episodes_before_start']} initial episodes...")
n_init = config['episodes_before_start']
for _ in range(n_init // 2):
    collect_episode(env, rssm, rssm.buffer, action_fn=scripted_policy)
    collect_episode(env, rssm, rssm.buffer)
if n_init % 2:
    collect_episode(env, rssm, rssm.buffer, action_fn=scripted_policy)
print(f"Buffer size: {len(rssm.buffer)}")


## 4. Reconstruction visualization helper

Decodes latent states back to pixels and compares with ground truth observations.

In [ ]:
@torch.no_grad()def visualize_reconstructions(rssm, buffer, num_samples=4, save_path=None):    data = buffer.sample(num_samples, 2)    obs = data['observations']    obs_flat = obs[:, 1].to(rssm.device)    encoded = rssm.encoder(obs_flat)    recurrent = torch.zeros(num_samples, rssm.recurrent_size, device=rssm.device)    latent, _ = rssm.posterior_net(torch.cat((recurrent, encoded), -1))    full_state = torch.cat((recurrent, latent), -1)    recon = rssm.decoder(full_state)    obs_np = obs_flat.cpu().numpy()    recon_np = recon.cpu().numpy()    fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 4))    for i in range(num_samples):        axes[0, i].imshow(np.transpose(obs_np[i], (1, 2, 0)))        axes[0, i].set_title('Original')        axes[0, i].axis('off')        axes[1, i].imshow(np.transpose(recon_np[i].clip(0, 1), (1, 2, 0)))        axes[1, i].set_title('Reconstruction')        axes[1, i].axis('off')    plt.tight_layout()    if save_path:        os.makedirs(os.path.dirname(save_path), exist_ok=True)        plt.savefig(save_path, bbox_inches='tight', dpi=120)    plt.close(fig)

## 5. Training loop

Alternates between gradient steps and environment interaction.
Shows live loss curves, periodic reconstructions, and evaluation rewards.

In [ ]:
iters = config['gradient_steps'] // config['replay_ratio']loss_history = {'wm_loss': [], 'recon_loss': [], 'prior_recon_loss': [], 'reward_loss': [], 'kl_raw': [], 'kl_active': []}eval_history = []os.makedirs('plots', exist_ok=True)pbar = tqdm(range(iters), desc='Training')for iteration in pbar:    for _ in range(config['replay_ratio']):        data = rssm.buffer.sample(rssm_cfg['batch_size'], rssm_cfg['batch_length'])        _, metrics = rssm.train_step(data)        rssm.total_gradient_steps += 1    for k in loss_history:        loss_history[k].append(metrics[k])    n_interact = config.get('num_interaction_episodes', 1)
    for _ in range(n_interact // 2):
        collect_episode(env, rssm, rssm.buffer, action_fn=scripted_policy)
        collect_episode(env, rssm, rssm.buffer)
    if n_interact % 2:
        collect_episode(env, rssm, rssm.buffer, action_fn=scripted_policy)
    if iteration % 10 == 0 and iteration > 0:        tqdm.write(            f"[{rssm.total_gradient_steps:>6d}] "            f"wm={metrics['wm_loss']:.2f} "            f"recon={metrics['recon_loss']:.2f} "            f"prior_recon={metrics['prior_recon_loss']:.4f} "            f"reward={metrics['reward_loss']:.2f} nonzero={metrics['reward_nonzero']:.3f} "            f"kl_raw={metrics['kl_raw']:.4f}  "            f"kl_act={metrics['kl_active']:.2f}\n"
            f"buf={len(rssm.buffer)}"        )        gs = rssm.total_gradient_steps        fig, axes = plt.subplots(2, 4, figsize=(16, 8))        colors = {'wm_loss': 'blue', 'recon_loss': 'green', 'reward_loss': 'orange', 'kl_raw': 'red', 'kl_active': 'purple'}        for ax, (k, color) in zip(axes.flat[:5], colors.items()):            ax.plot(loss_history[k], color=color)            ax.set_title(k)            ax.set_xlabel('Iteration')        if eval_history:            steps, means, stds = zip(*eval_history)            axes.flat[5].errorbar(steps, means, yerr=stds, capsize=3)            axes.flat[5].set_title('Eval reward')            axes.flat[5].set_xlabel('Gradient step')        axes.flat[6].text(0.5, 0.5, 'Reconstructions below \u2193',                          ha='center', va='center', fontsize=14)        axes.flat[6].axis('off')        plt.tight_layout()        plt.savefig(f'plots/loss_{gs:06d}.png', bbox_inches='tight', dpi=120)        plt.close(fig)        if iteration % 50 == 0:            visualize_reconstructions(rssm, rssm.buffer, num_samples=4,                                      save_path=f'plots/recon_{gs:06d}.png')    gs = rssm.total_gradient_steps    if config['save_checkpoints'] and gs % config['checkpoint_interval'] == 0:        avg, std = evaluate(            env_eval, rssm,            lambda s, obs=None: torch.nn.functional.one_hot(                torch.randint(0, action_size, (1,), device=device), action_size            ).float(),            num_episodes=config['num_evaluation_episodes'],        )        eval_history.append((gs, avg, std))        path = (            f"{config['folder_names']['checkpoints_folder']}/"            f"{config['environment_name']}_{config['run_name']}_{gs // 1000}k"        )        rssm.save_checkpoint(path)        pbar.set_postfix(wm_loss=f"{metrics['wm_loss']:.2f}",                         eval_reward=f"{avg:.2f}\u00b1{std:.2f}")        tqdm.write(            f"Step {gs:>6d} | WM loss: {metrics['wm_loss']:.2f} | "            f"Eval reward: {avg:.2f} \u00b1 {std:.2f}"        )

## 6. Save training metrics

In [ ]:
df_loss = pd.DataFrame(loss_history)
df_loss.to_csv('training_metrics.csv', index=False)
print(f'Loss history saved: {len(df_loss)} rows')

if eval_history:
    df_eval = pd.DataFrame(eval_history, columns=['gradient_step', 'mean_reward', 'std_reward'])
    df_eval.to_csv('eval_metrics.csv', index=False)
    print(f'Eval history saved: {len(df_eval)} rows')

final_path = (
    f"{config['folder_names']['checkpoints_folder']}/"
    f"{config['environment_name']}_{config['run_name']}_final"
)
rssm.save_checkpoint(final_path)
print(f'Final checkpoint saved: {final_path}.pth')

env.close()
env_eval.close()
print('Done.')

## 7. Visualize imagined trajectory

Encode a real observation from the buffer, then rollout the prior
with random actions for 15 steps. Decode each imagined state back
to pixels and compare with the real trajectory.

In [ ]:
@torch.no_grad()
def visualize_rollout(rssm, buffer, device, action_size, horizon=15, save_path=None):
    data = buffer.sample(1, horizon + 1)
    real_obs = data['observations'][0, 1:]

    obs_0 = data['observations'][0, 0].to(device)
    recurrent = torch.zeros(1, rssm.recurrent_size, device=device)
    latent = torch.zeros(1, rssm.latent_size, device=device)
    action = torch.zeros(1, action_size, device=device)

    recurrent, latent = rssm.encode_step(
        recurrent, latent, action, obs_0.cpu().numpy()
    )

    random_action_fn = lambda s: torch.nn.functional.one_hot(
        torch.randint(0, action_size, (1,), device=device), action_size
    ).float()

    imagined = rssm.rollout_prior(recurrent, latent, random_action_fn, horizon)
    prior_decoded = rssm.decoder(imagined.view(-1, rssm.full_state_size))
    prior_decoded = prior_decoded.view(horizon, *obs_shape)

    # ── Posterior reconstruction ──
    post_decoded = []
    rec, lat = recurrent.clone(), latent.clone()
    for t in range(horizon):
        full = torch.cat((rec, lat), -1)
        post_decoded.append(rssm.decoder(full).squeeze(0))
        if t < horizon - 1:
            enc_obs = rssm.encoder(real_obs[t].to(device).unsqueeze(0))
            rec = rssm.recurrent_model(rec, lat, data['actions'][0, t].unsqueeze(0).to(device))
            lat, _ = rssm.posterior_net(torch.cat((rec, enc_obs), -1))
    post_decoded = torch.stack(post_decoded)

    fig, axes = plt.subplots(3, horizon, figsize=(horizon * 2, 6))
    for t in range(horizon):
        axes[0, t].imshow(np.transpose(real_obs[t].cpu().numpy(), (1, 2, 0)))
        axes[0, t].axis('off')
        if t == 0:
            axes[0, t].set_title('Real', fontsize=10)

        axes[1, t].imshow(np.transpose(post_decoded[t].cpu().numpy().clip(0, 1), (1, 2, 0)))
        axes[1, t].axis('off')
        if t == 0:
            axes[1, t].set_title('Posterior', fontsize=10)

        axes[2, t].imshow(np.transpose(prior_decoded[t].cpu().numpy().clip(0, 1), (1, 2, 0)))
        axes[2, t].axis('off')
        if t == 0:
            axes[2, t].set_title('Imagined (prior)', fontsize=10)

    plt.suptitle(f'Real vs Posterior vs Imagined (horizon={horizon})')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.close(fig)
os.makedirs('visualizations', exist_ok=True)for i in range(5):    visualize_rollout(        rssm, rssm.buffer, device, action_size, horizon=15,        save_path=f'visualizations/rollout_{i:02d}.png',    )